# OpenRouter deep test — `stealth/ox-alpha`

End-to-end tests for the OpenRouter OpenAI-compatible endpoint: **text, streaming, vision, tool-calling (single + multi-arg), the shipit adapter, and a multi-agent example.**

> Run cells top-to-bottom. The key is entered securely at runtime (never saved in the notebook).
> Model note: `stealth/ox-alpha` is a fast text/vision model but **does not accept video URLs** (returns 404). Images and tools work.


## 0 · Setup


In [ ]:
# pip install openai  (already present in most envs)
import os, json, time, getpass
from openai import OpenAI

# Enter your OpenRouter key securely (not stored in the notebook).
API_KEY = os.getenv('OPENROUTER_API_KEY') or getpass.getpass('OpenRouter API key: ')
MODEL   = 'stealth/ox-alpha'

client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=API_KEY,
                timeout=120, max_retries=1)
HEADERS = {'HTTP-Referer': 'https://aftrdrk.dev', 'X-Title': 'OpenRouter deep test'}
print('client ready ·', MODEL)


## 1 · Basic text + latency


In [ ]:
t0 = time.time()
r = client.chat.completions.create(
    model=MODEL, extra_headers=HEADERS, max_tokens=60,
    messages=[{'role':'user','content':'In one sentence, what is a ransomware double-extortion attack?'}])
print('latency: %.1fs' % (time.time()-t0))
print('content:', r.choices[0].message.content)
print('usage  :', r.usage.model_dump())


## 2 · Streaming (token-by-token)


In [ ]:
stream = client.chat.completions.create(
    model=MODEL, extra_headers=HEADERS, max_tokens=120, stream=True,
    messages=[{'role':'user','content':'List 3 detection tips for the Akira ransomware group, briefly.'}])
for chunk in stream:
    d = chunk.choices[0].delta.content if chunk.choices else None
    if d: print(d, end='')
print('\n\n[stream done]')


## 3 · Vision (image)
`stealth/ox-alpha` can read images. It can be slow (30-90s) — be patient.


In [ ]:
t0 = time.time()
r = client.chat.completions.create(
    model=MODEL, extra_headers=HEADERS, max_tokens=150,
    messages=[{'role':'user','content':[
        {'type':'text','text':'What is in this image? One sentence.'},
        {'type':'image_url','image_url':{'url':'https://live.staticflickr.com/3851/14825276609_098cac593d_b.jpg'}},
    ]}])
print('latency: %.1fs' % (time.time()-t0))
print(r.choices[0].message.content)


## 4 · Tool calling — single tool
Native OpenAI function-calling. The model should return a structured `tool_call`.


In [ ]:
tools = [{'type':'function','function':{
    'name':'get_weather','description':'Current weather for a city',
    'parameters':{'type':'object','properties':{'city':{'type':'string'}},'required':['city']}}}]
r = client.chat.completions.create(
    model=MODEL, extra_headers=HEADERS, tools=tools, tool_choice='auto', max_tokens=100,
    messages=[{'role':'user','content':"What's the weather in Berlin?"}])
m = r.choices[0].message
print('tool_calls:', [(t.function.name, t.function.arguments) for t in (m.tool_calls or [])] or 'none')
print('text      :', m.content)


## 5 · Multi-argument tool (MCP-style)
The real agent test: can it fill several arguments in one structured call?


In [ ]:
tools = [{'type':'function','function':{
    'name':'query_records','description':'Query internal records with multiple filters',
    'parameters':{'type':'object','properties':{
        'domain':{'type':'string','description':'e.g. ransomlook'},
        'entity':{'type':'string','description':'e.g. groups'},
        'entity_id':{'type':'string','description':'e.g. akira'}},
    'required':['domain','entity','entity_id']}}}]
r = client.chat.completions.create(
    model=MODEL, extra_headers=HEADERS, tools=tools, tool_choice='required', max_tokens=120,
    messages=[{'role':'user','content':'Pull the RansomLook group profile for Akira.'}])
m = r.choices[0].message
for t in (m.tool_calls or []):
    print(t.function.name, '->', json.loads(t.function.arguments))


## 6 · Through the shipit adapter (`OpenAIChatLLM`)
Same call, but via shipit's OpenAI-compatible adapter — proving OpenRouter works inside the agent framework.
Requires `shipit_agent` importable (run in an env where it's installed).


In [ ]:
try:
    from shipit_agent.llms.openai_adapter import OpenAIChatLLM
    from shipit_agent.models import Message
    llm = OpenAIChatLLM(model=MODEL, api_key=API_KEY,
                        base_url='https://openrouter.ai/api/v1',
                        default_headers=HEADERS)
    resp = llm.complete(messages=[Message(role='user', content='Name one famous ransomware group in 2 words.')], tools=None)
    print('content:', resp.content)
    print('usage  :', resp.usage)   # cache_read_input_tokens surfaces here when warm
except ImportError as e:
    print('shipit_agent not importable in this kernel — skip:', e)


## 7 · Multi-agent example
A tiny two-agent pipeline on OpenRouter: **Researcher** proposes findings, **Critic** reviews them, **Editor** writes the final brief. Pure OpenAI-compatible calls — swap in real tools/MCP as needed.


In [ ]:
def ask(system, user, max_tokens=300):
    r = client.chat.completions.create(
        model=MODEL, extra_headers=HEADERS, max_tokens=max_tokens, temperature=0.3,
        messages=[{'role':'system','content':system},{'role':'user','content':user}])
    return r.choices[0].message.content or ''

TOPIC = 'the Akira ransomware group and its threat to manufacturers'

print('── RESEARCHER ──')
research = ask('You are a threat-intel researcher. Be concrete and factual.',
               f'Give 5 key findings about {TOPIC}. Bullet points.')
print(research)

print('\n── CRITIC ──')
critique = ask('You are a skeptical reviewer. Flag anything unsupported or vague.',
               f'Review these findings and note weaknesses:\n{research}')
print(critique)

print('\n── EDITOR ──')
brief = ask('You are a senior analyst writing for a client board.',
            f'Using the findings and the critique, write a tight 1-paragraph brief.\n\nFINDINGS:\n{research}\n\nCRITIQUE:\n{critique}')
print(brief)


## 8 · Agentic tool loop (multi-turn, one agent + tools)
A minimal manual agent loop: the model calls a tool, we execute it, feed the result back, and it answers. This is the core of any MCP agent.


In [ ]:
def fake_lookup(domain, entity, entity_id):
    # stand-in for a real MCP tool
    return json.dumps({'domain':domain,'entity':entity,'id':entity_id,
        'first_seen':'2023-03','model':'RaaS','notable':'targets VMware ESXi'})

tools = [{'type':'function','function':{'name':'query_records',
    'parameters':{'type':'object','properties':{'domain':{'type':'string'},
    'entity':{'type':'string'},'entity_id':{'type':'string'}},
    'required':['domain','entity','entity_id']},'description':'Query internal records'}}]

msgs = [{'role':'user','content':'Look up the Akira group profile in ransomlook and summarize it.'}]
for step in range(4):
    r = client.chat.completions.create(model=MODEL, extra_headers=HEADERS,
        tools=tools, tool_choice='auto', max_tokens=300, messages=msgs)
    m = r.choices[0].message
    if m.tool_calls:
        msgs.append({'role':'assistant','content':m.content or '','tool_calls':[t.model_dump() for t in m.tool_calls]})
        for t in m.tool_calls:
            args = json.loads(t.function.arguments)
            result = fake_lookup(**args)
            print(f'[tool] {t.function.name}({args}) -> {result[:60]}...')
            msgs.append({'role':'tool','tool_call_id':t.id,'content':result})
    else:
        print('\n[final answer]\n' + (m.content or ''))
        break


## 9 · Latency / cache benchmark
Send the same large prefix 3x back-to-back and watch `cached_tokens` climb (implicit caching).


In [ ]:
PREFIX = 'Background: ' + ('Akira is a ransomware group. ' * 300)
for i in range(3):
    t0 = time.time()
    r = client.chat.completions.create(model=MODEL, extra_headers=HEADERS, max_tokens=8,
        messages=[{'role':'user','content':PREFIX + f'\n\nSay the number {i}.'}])
    u = r.usage.model_dump(); pd = u.get('prompt_tokens_details',{})
    print(f'call {i}: {time.time()-t0:.1f}s · prompt={u["prompt_tokens"]} · cached={pd.get("cached_tokens")}')
